<a href="https://colab.research.google.com/github/ncrowder/maven/blob/main/maven_drill_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Data here: https://mavenanalytics.io/data-drills/estimate-the-estate
df = pd.read_csv('manhattan_property_sales.csv')

In [ ]:
df.head()

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE
0,CHELSEA,150 WEST 15TH STREET,10011,A4,8997,14999999
1,CHELSEA,348 WEST 22 STREET,10011,A4,3042,13500000
2,CHELSEA,204 WEST 21ST STREET,10011,A4,3365,7650000
3,CHELSEA,481 WEST 22 STREET,10011,A4,3666,0
4,CHELSEA,344 W 22 STREET,10011,A4,4395,13100000


## df Info & Describe

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141 entries, 0 to 140
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   NEIGHBORHOOD    141 non-null    object
 1   ADDRESS         141 non-null    object
 2   ZIP_CODE        141 non-null    int64 
 3   BUILDING_CLASS  141 non-null    object
 4   SQUARE_FEET     141 non-null    int64 
 5   SALE_PRICE      141 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 6.7+ KB


In [ ]:
df.describe()

,ZIP_CODE,SQUARE_FEET,SALE_PRICE
count,141.000000,141.000000,1.410000e+02
mean,10052.000000,4601.014184,8.401532e+06
std,79.131626,2489.622682,9.465731e+06
min,10003.000000,804.000000,0.000000e+00
25%,10016.000000,3038.000000,2.500000e+06
50%,10024.000000,3928.000000,6.500000e+06
75%,10065.000000,5712.000000,1.070000e+07
max,10463.000000,17676.000000,5.500000e+07


In [ ]:
df.value_counts(subset=['ZIP_CODE','BUILDING_CLASS'])

,,count
ZIP_CODE,BUILDING_CLASS,
10021,A4,15
10014,A4,14
10065,A4,13
10128,A4,11
10011,A4,9
10024,A4,8
10075,A4,7
10028,A4,7
10003,A4,5


## My Solution

In [ ]:
df_price = df.assign(
    priceper = df['SALE_PRICE'] / df['SQUARE_FEET']
)

def market_val(row):

    if row['SALE_PRICE'] != 0:
        return row['SALE_PRICE']

    zipcode = row['ZIP_CODE']
    buildclass = row['BUILDING_CLASS']

    avgpriceper = (
        df_price[
            (df_price['ZIP_CODE'] == zipcode) &
            (df_price['BUILDING_CLASS'] == buildclass) &
            (df_price['priceper'] > 0) # to protect against averaging in rows whose saleprice is $0 within that same zipcode and build class
        ]['priceper'].mean().astype(int)
    )
    return row['SQUARE_FEET'] * avgpriceper

In [ ]:
df['MARKET_VALUE'] = df.apply(market_val,axis=1)

In [ ]:
df

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE,MARKET_VALUE_2,MARKET_VALUE
0,CHELSEA,150 WEST 15TH STREET,10011,A4,8997,14999999,14999999,14999999
1,CHELSEA,348 WEST 22 STREET,10011,A4,3042,13500000,13500000,13500000
2,CHELSEA,204 WEST 21ST STREET,10011,A4,3365,7650000,7650000,7650000
3,CHELSEA,481 WEST 22 STREET,10011,A4,3666,0,9780888,9780888
4,CHELSEA,344 W 22 STREET,10011,A4,4395,13100000,13100000,13100000
...,...,...,...,...,...,...,...,...
136,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,3500000,3500000,3500000
137,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,0,5797248,5797248
138,UPPER WEST SIDE,52 WEST 84TH STREET,10024,A5,3651,0,5816043,5816043
139,UPPER WEST SIDE,315 WEST 84 STREET,10024,A5,4078,6500000,6500000,6500000


In [ ]:
df[df['MARKET_VALUE'] > 15_000_000].shape[1]

8

##

 ## Optimized solution (using ChatGPT)

In [ ]:
df_price = df.assign(
    priceper = df['SALE_PRICE'] / df['SQUARE_FEET']
)

avg_price = (
    df_price[df_price['priceper'] > 0]
    .groupby(['ZIP_CODE', 'BUILDING_CLASS'])['priceper']
    .mean()
)

df['MARKET_VALUE_2'] = np.where(
    df['SALE_PRICE'] != 0,
    df['SALE_PRICE'],
    df['SQUARE_FEET'] * df.set_index(
        ['ZIP_CODE', 'BUILDING_CLASS']
    ).index.map(avg_price).astype(int)
)

In [ ]:
df

,NEIGHBORHOOD,ADDRESS,ZIP_CODE,BUILDING_CLASS,SQUARE_FEET,SALE_PRICE,MARKET_VALUE_2
0,CHELSEA,150 WEST 15TH STREET,10011,A4,8997,14999999,14999999
1,CHELSEA,348 WEST 22 STREET,10011,A4,3042,13500000,13500000
2,CHELSEA,204 WEST 21ST STREET,10011,A4,3365,7650000,7650000
3,CHELSEA,481 WEST 22 STREET,10011,A4,3666,0,9780888
4,CHELSEA,344 W 22 STREET,10011,A4,4395,13100000,13100000
...,...,...,...,...,...,...,...
136,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,3500000,3500000
137,UPPER WEST SIDE,324 WEST 85 STREET,10024,A4,2976,0,5797248
138,UPPER WEST SIDE,52 WEST 84TH STREET,10024,A5,3651,0,5816043
139,UPPER WEST SIDE,315 WEST 84 STREET,10024,A5,4078,6500000,6500000


In [ ]:
df[df['MARKET_VALUE_2'] > 15_000_000].shape[1]

7